In [ ]:
from pathlib import Path
import sys
import json
from datetime import datetime

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Llm.llm_loader import LLM
import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv(project_root / '.env')

In [ ]:
model_name = 'llama3.1:8b'
Answer_llm = LLM(model_name = "llama3.1:8b", provider= "ollama")
dataset = 'MMQA'
prompt_path = project_root / 'Templates' / 'zero_shot' / 'baseline_schema_linking.txt'
logs_dir = project_root / 'Logs' / model_name
logs_dir.mkdir(exist_ok=True)
run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
log_path = logs_dir / f'zero_shot_baseline_schema_linking_{dataset}_{run_id}.json'

In [ ]:
if dataset == 'MMQA':
    datasets = [
        (
            'two_table',
            project_root / 'Data' / dataset / 'Synthesized_two_table_with_spider_db_id.json',
            pd.read_json(project_root / 'Data' / dataset / 'Synthesized_two_table_with_spider_db_id.json')
        ),
        (
            'three_table',
            project_root / 'Data' / dataset / 'Synthesized_three_table_with_spider_db_id.json',
            pd.read_json(project_root / 'Data' / dataset / 'Synthesized_three_table_with_spider_db_id.json')
        ),
    ]
else:
    raise ValueError(f'Unsupported dataset: {dataset}')

In [ ]:
def append_log_entry(log_records, data_split, row, response_text):
    log_records.append(
        {
            'model': Answer_llm.model_name,
            'provider': Answer_llm.provider,
            'template_name': prompt_path.name,
            'id': f"{dataset}-{data_split}-{row['id_']}",
            'spider_db_id': row['Spider_db_id'],
            'question': row['Question'],
            'response': response_text,
        }
    )
    log_path.write_text(json.dumps(log_records, ensure_ascii=False, indent=2), encoding='utf-8')

In [ ]:
prompt_template = prompt_path.read_text(encoding='utf-8').strip()
log_records = []
log_path.write_text(json.dumps(log_records, ensure_ascii=False, indent=2), encoding='utf-8')

for data_split, data_source, df in datasets:
    for _, row in tqdm(df.iterrows(), total=len(df), desc=data_split):
        schema_path = project_root / 'Data' / 'spider' / 'database' / row['Spider_db_id'] / 'schema description.csv'
        schema_df = pd.read_csv(schema_path)
        database_schema = schema_df.to_markdown()
        prompt = (
            prompt_template
            .replace('{DATABASE_SCHEMA}', database_schema)
            .replace('{QUESTION}', row['Question'])
            .replace('{HINT}', 'No hint')
        )
        res = Answer_llm.query(prompt)
        append_log_entry(log_records, data_split, row, res)

print(f'All responses have been saved to {log_path}')